In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
from sklearn.preprocessing import MinMaxScaler
from wifiplotting import *
from modeling import *
import dill

sequoia = pd.read_csv('../data/sequoia_sets.csv')

TL_CORNER = [37.430582, -122.173904]
BR_CORNER = [37.42705, -122.169413]

In [ ]:
osm_context = OSMPlotContext.from_bounds(
    init_lons=[BR_CORNER[1], TL_CORNER[1]], init_lats=[BR_CORNER[0], TL_CORNER[0]],
    pad_fraction=0.0
)

with open('data/osm_context.pkl', 'wb') as f:
    dill.dump(osm_context, f)

In [ ]:
sequoia_nonan = sequoia[sequoia['rssi_set'].notna()].copy().reset_index()
time0 = dt.strptime(sequoia['timestamp'][0], "%Y-%m-%d %H:%M:%S.%f%z")
sequoia_nonan['t'] = sequoia_nonan['timestamp'].apply(
    lambda x: (dt.strptime(x, "%Y-%m-%d %H:%M:%S.%f%z") - time0).total_seconds() / (60 * 60))

scaler = MinMaxScaler()
scaler.fit(pd.DataFrame(np.stack([TL_CORNER[::-1], BR_CORNER[::-1]]), columns=['longitude', 'latitude']))
# scaler.fit(sequoia_nonan[['longitude', 'latitude']])
X_train = pd.DataFrame()
X_train[['longitude', 'latitude']] = scaler.transform(sequoia_nonan[['longitude', 'latitude']])
X_train['indoor'] = sequoia_nonan['indoor'].astype(float).values
X_train['t'] = sequoia_nonan['t'].values

y_train = sequoia_nonan['rssi_set']

X_train = np.array(X_train)
y_train = np.array(y_train)

np.save('data/X_train.npy', X_train)
np.save('data/y_train.npy', y_train)
np.save('data/obs_count.npy', sequoia_nonan['rssi_n_valid'].to_numpy())
np.save('data/obs_sse.npy', sequoia_nonan['rssi_within_sse'].fillna(0.0).to_numpy())

In [ ]:
np.save('data/coord_train.npy', sequoia_nonan[['longitude', 'latitude']].to_numpy())

wlon_train, wlat_train = osm_context.to_world(sequoia_nonan.longitude, sequoia_nonan.latitude)

np.save('data/world_train.npy', np.stack([wlon_train, wlat_train]).T)

In [ ]:
grid_width = 100

x_new = np.linspace(0, 1, grid_width)
y_new = np.linspace(0, 1, grid_width)

grid_points = np.stack(np.meshgrid(x_new, y_new), axis=-1).reshape(-1, 2)

coord_grid = scaler.inverse_transform(np.concatenate([grid_points], axis=-1))
long_new, lat_new = coord_grid.T
wlon_test, wlat_test = osm_context.to_world(long_new, lat_new)

z_new = osm_context.contains_building(long_new, lat_new).reshape(-1,1)
t_new = np.ones((grid_points.shape[0], 1)) * 1000

X_new = np.concatenate([grid_points, z_new, t_new], axis=-1)

np.save('data/X_test.npy', X_new)
np.save('data/coord_test.npy', coord_grid)
np.save('data/world_test.npy', np.stack([wlon_test, wlat_test]).T)

In [ ]:
geo_train, geo_val = geographic_train_test_split(sequoia_nonan)
geo_tr_idx = geo_train.index
geo_val_idx = geo_val.index

np.save('data/geo_X_train.npy', X_train[geo_tr_idx])
np.save('data/geo_y_train.npy', y_train[geo_tr_idx])
np.save('data/geo_X_val.npy', X_train[geo_val_idx])
np.save('data/geo_y_val.npy', y_train[geo_val_idx])